# 흐린 위성사진을 선명하게 만들기 (EDSR ×3)

같은 장소를 두 위성이 찍었다. 하나는 흐리고, 하나는 선명하다.

| | 위성 | 한 픽셀이 담는 실제 크기 |
|---|---|---|
| 흐린 쪽 | Sentinel-2 | 10 m |
| 선명한 쪽 | IKONOS | 3.33 m |

10 m짜리를 3.33 m짜리처럼 보이게 만드는 게 목표다. 가로세로 3배씩 키우는 셈이라
**×3 초해상화**라고 부른다. 그냥 확대하면 뿌옇게 뭉개지니까, 선명한 사진을 정답으로 주고
"이런 식으로 채워라"를 AI에게 가르친다.

## 흐린 사진을 만드는 두 가지 방법

이 실습에는 흐린 사진이 두 종류 나온다. **어떻게 만들었는지가 다르다.**

- **줄여서 만든 것** — 선명한 정답 사진을 3배 축소한 것. 정답에서 나왔으니 색과 밝기가
  정확히 맞는다. 연습문제와 시험문제가 전부 이것으로 되어 있다.
- **진짜 찍은 것** — Sentinel-2 위성이 실제로 촬영한 사진. 마지막 인천 사진이 이것이다.

앞의 것으로 배우고 시험 보다가, 마지막에 진짜 사진에 써보는 흐름이다.

## 준비물

| | 개수 | 무엇 |
|---|---|---|
| 연습문제 | 40쌍 | 흐린 128픽셀 → 선명한 384픽셀 |
| 시험문제 | 10쌍 | 같은 방식. 단 **배울 때 안 본 지역**에서만 |
| 최종 테스트 | 1장 | 인천 사진. 진짜 위성 촬영본, 정답이 없다 |

시험문제가 연습문제와 같은 방식인 이유는, **"안 배운 지역에도 통하는가"** 하나만
깨끗하게 보기 위해서다. 사진 만드는 방식까지 다르면 점수가 왜 낮은지 알 수 없게 된다.

## 이 노트북에서 하는 일

1. 자료를 받고 눈으로 확인한다
2. **학습 코드가 제대로 도는지 짧게 돌려서 확인한다** (3회, 몇 초)
3. **미리 제대로 학습해둔 가중치**로 인천 사진을 처리한다
4. 그냥 확대한 것과 비교한다

2번은 어디까지나 동작 확인이다. 몇 초 학습으로는 쓸 만한 결과가 안 나오니,
실제 결과는 3번에서 만들어둔 가중치로 본다.

> **먼저 GPU를 켜세요**: 위 메뉴에서 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장

## 0. GPU가 켜져 있나 확인

GPU 없이도 돌아가긴 하지만 몇십 배 느리다. 아래를 실행해서 `True`가 나와야 한다.

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  ', torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU가 꺼져 있습니다. 런타임 → 런타임 유형 변경 → T4 GPU')

## 1. 실습 자료 가져오기

Colab은 구글이 잠깐 빌려주는 컴퓨터다. **내 컴퓨터의 파일은 안 보인다.** 그래서 매번
자료를 인터넷에서 받아와야 한다.

기본값(`'github'`)이면 아무것도 안 고쳐도 된다. 그냥 실행하면 된다.

In [ ]:
import os, shutil, subprocess, glob

SOURCE      = 'github'                                        # github | drive | url
GITHUB_REPO = 'https://github.com/BWMIN-Hub/SR_practice.git'  # SOURCE='github' 일 때
DRIVE_ZIP   = '/content/drive/MyDrive/sr_colab/colab.zip'     # SOURCE='drive' 일 때
BUNDLE_URL  = ''                                              # SOURCE='url' 일 때


def _mount_drive():
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')


def _find_root(base):
    """dataset/ models/ notebooks/ 를 모두 가진 폴더를 찾는다.

    저장소 루트가 colab/ 자체든, colab/ 을 품은 상위 폴더든 양쪽 다 찾아낸다.
    """
    for d, subs, _ in os.walk(base):
        if {'dataset', 'models', 'notebooks'} <= set(subs):
            return d
    return None


def fetch():
    """번들을 /content 아래로 가져오고 그 루트 경로를 돌려준다."""
    hit = _find_root('/content')
    if hit:                                      # 이미 있으면 다시 받지 않는다
        return hit

    if SOURCE == 'github':
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, '/content/repo'],
                       check=True)
    elif SOURCE in ('drive', 'url'):
        zp = '/content/colab.zip'
        if SOURCE == 'drive':
            _mount_drive()
            assert os.path.exists(DRIVE_ZIP), f'Drive에 없습니다: {DRIVE_ZIP}'
            shutil.copy(DRIVE_ZIP, zp)           # 순차 읽기 1회
        else:
            subprocess.run(['wget', '-q', '-O', zp, BUNDLE_URL], check=True)
        subprocess.run(['unzip', '-q', '-o', zp, '-d', '/content/repo'], check=True)
    else:
        raise ValueError(SOURCE)

    hit = _find_root('/content/repo')
    assert hit, '받은 내용에서 dataset/·models/·notebooks/ 를 가진 폴더를 찾지 못했습니다'
    return hit


ROOT  = fetch()
DATA  = f'{ROOT}/dataset'
MODEL = f'{ROOT}/models/01_edsr_x3'
CODE  = f'{MODEL}/code'
assert os.path.isdir(CODE), CODE
print('번들 준비 완료:', ROOT)

### 결과물을 남기고 싶다면 (선택)

Colab은 **12시간이 지나거나 90분쯤 가만히 두면 연결이 끊긴다.** 그러면 그 안에 있던
파일이 전부 사라진다.

이 실습은 몇 분이면 끝나서 **기본값은 꺼둔 상태**다. 결과를 남기고 싶을 때만 아래
`SAVE_TO_DRIVE`를 `True`로 바꾸면, 학습 결과 폴더가 구글 드라이브에 연결된다.
(켜면 드라이브 접근 권한을 묻는 창이 뜬다.)

In [ ]:
SAVE_TO_DRIVE = False   # 결과를 구글 드라이브에 남기려면 True

if SAVE_TO_DRIVE:
    _mount_drive()
    dst = '/content/drive/MyDrive/sr_colab/experiment'
    os.makedirs(dst, exist_ok=True)
    if not os.path.islink(f'{CODE}/experiment'):
        shutil.rmtree(f'{CODE}/experiment', ignore_errors=True)
        os.symlink(dst, f'{CODE}/experiment')
    print('experiment ->', os.path.realpath(f'{CODE}/experiment'))
else:
    os.makedirs(f'{CODE}/experiment', exist_ok=True)

### 필요한 프로그램 설치

거의 다 Colab에 이미 깔려 있다. 위성사진(GeoTIFF)을 읽는 `rasterio` 하나만 받으면 된다.

그래프 라벨은 영어로 써두었다. 한글로 쓰면 Colab에 한글 폰트가 없어 네모로 깨지고,
폰트를 설치하면 그것만으로 30~60초가 더 걸린다.

In [ ]:
!pip install -q rasterio

import matplotlib as mpl
import rasterio

mpl.rc('axes', unicode_minus=False)
# 입력 크기가 일정하므로 켜두면 conv 알고리즘을 한 번 고르고 재사용한다
torch.backends.cudnn.benchmark = True

print('rasterio', rasterio.__version__, '| cudnn.benchmark', torch.backends.cudnn.benchmark)

## 2. 자료 살펴보기

무엇을 가지고 시작하는지 먼저 눈으로 본다.

In [ ]:
from collections import Counter

for split in ['training', 'validation']:
    hr = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))
    lr = sorted(glob.glob(f'{DATA}/{split}/LR_bicubic/X3/*.png'))
    city = Counter(os.path.basename(f).rsplit('_y', 1)[0].rsplit('_', 1)[0] for f in hr)
    print(f'{split:11s} HR {len(hr):3d}장 / LR {len(lr):3d}장   {dict(city)}')

print('\ntest      ', [os.path.basename(f) for f in glob.glob(f'{DATA}/test/*.tif')])

In [ ]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt

def preview(split, n=4):
    hr_files = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))[:n]
    fig, ax = plt.subplots(2, len(hr_files), figsize=(3.2 * len(hr_files), 6.8))
    for i, f in enumerate(hr_files):
        stem = os.path.basename(f)[:-4]
        hr = imageio.imread(f)
        lr = imageio.imread(f'{DATA}/{split}/LR_bicubic/X3/{stem}x3.png')
        ax[0, i].imshow(lr); ax[0, i].set_title(f'LR {lr.shape[1]}x{lr.shape[0]}', fontsize=9)
        ax[1, i].imshow(hr); ax[1, i].set_title(f'HR {hr.shape[1]}x{hr.shape[0]}', fontsize=9)
        ax[1, i].set_xlabel(stem.rsplit('_y', 1)[0], fontsize=7)
        for a in (ax[0, i], ax[1, i]): a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'{split}   top: input LR 10 m   /   bottom: target HR 3.33 m')
    plt.tight_layout(); plt.show()

preview('training')
preview('validation')

### 뭘 보면 되나

위가 흐린 입력, 아래가 선명한 정답이다. **위를 아래처럼 만드는 게 이 AI가 할 일이다.**

연습문제(training)와 시험문제(validation)는 **만드는 방식이 같다.** 둘 다 정답 사진을
3배 줄여서 흐리게 만들었다. 다른 건 **지역**뿐이다 — 시험문제는 배울 때 한 번도 안 본
도시 구역에서 가져왔다.

그래서 시험 점수는 "안 배운 곳에서도 통하는가"를 그대로 보여준다.

## 3. 학습 코드 돌려보기

여기서는 **코드가 정상적으로 도는지만 확인한다.** 3회만 돌리기 때문에 실력이 늘지는
않는다. 제대로 된 결과는 다음 단계에서 미리 학습해둔 가중치로 본다.

밑바닥부터 가르치면 몇 시간이 걸린다. 그래서 이미 학습된 상태에서 이어서 돌린다.

설정값은 아래 몇 개만 알면 된다.

| 값 | 뜻 |
|---|---|
| `EPOCHS='4'` | 문제집을 몇 번 반복할지. **4를 넣으면 실제로는 3번 돈다** (원본 코드의 버릇) |
| `TEST_EVERY='50'` | 한 번 돌 때 문제를 몇 개 풀지 (16 × 50 = 800개) |
| `LR='1e-4'` | 한 번에 얼마나 크게 고쳐잡을지 |
| `RESET='0'` | 연결이 끊겼을 때 이어서 하고 싶으면 이걸로 바꾼다 |

오래 학습시켜 보고 싶으면 `EPOCHS`를 늘리면 된다. 다만 40장짜리 문제집이라
금방 한계에 부딪힌다.

In [ ]:
import time

env = dict(os.environ,
    GPU='0',
    DATA='COLAB',
    DIR_DATA=DATA,
    EPOCHS='4',             # 실제 3회. 코드가 도는지 확인하는 용도라 짧게 잡았다
    DECAY='2',
    LR='1e-4',
    TEST_EVERY='50',        # 한 회에 문제 800개 (16 x 50)
    PRINT_EVERY='10',       # TEST_EVERY 보다 작아야 기록이 남는다
    N_THREADS='2',          # Colab은 CPU가 2개뿐
    SAVE='edsr_colab_x3',
    SAVE_RESULTS='0',
    RESET='1',              # 이어서 학습할 때는 '0'
    PRETRAIN=f'{MODEL}/checkpoints/edsr_ikonos_x3_best.pt',
)

t0 = time.time()
p = subprocess.run(['bash', f'{CODE}/run_train.sh'], env=env,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout.splitlines():
    if 'Making a binary' in line or 'it/s]' in line: continue
    print(line)
print(f'\n소요 {time.time() - t0:.0f}초')

## 4. 기록 확인

학습이 남긴 기록을 그래프로 본다.

- **왼쪽 — 연습문제 점수**: 낮을수록 잘 푸는 것 (틀린 정도)
- **오른쪽 — 시험 점수**: 높을수록 잘 푸는 것 (PSNR, dB 단위)

3회밖에 안 돌렸으니 점 3개짜리 그래프다. **모양보다 숫자가 나온다는 것 자체가 확인 목적**이다.

In [ ]:
import numpy as np
import torch

EXP = f'{CODE}/experiment/edsr_colab_x3'
# EDSR 이 epoch 별 평균을 텐서로 저장해준다 (log.txt 를 정규식으로 긁는 것보다 안전)
loss = torch.load(f'{EXP}/loss_log.pt').flatten().numpy()
psnr = torch.load(f'{EXP}/psnr_log.pt').flatten().numpy()
ep = np.arange(1, len(loss) + 1)
best = int(np.argmax(psnr))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep, loss, 'o-')
ax[0].set_title('Training loss (lower is better)')
ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
ax[1].plot(ep, psnr, 'o-')
ax[1].plot(best + 1, psnr[best], 'r*', ms=15, label=f'best {psnr[best]:.3f} @ep{best+1}')
ax[1].set_title('Validation PSNR, unseen areas (higher is better)')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'연습문제 점수  {loss[0]:.2f} -> {loss[-1]:.2f}   ({len(loss)}회 학습)')
print(f'시험 점수      {psnr[best]:.3f} dB (최고, {best+1}회차) / {psnr[-1]:.3f} dB (마지막)')
print('\n3회만 돌렸으므로 점수가 거의 안 움직이는 것이 정상이다.')

### 점수가 거의 안 움직인다

정상이다. 이유가 두 가지다.

**1. 3회만 돌렸다.** 애초에 실력이 늘 만큼 돌리지 않았다.

**2. 출발점이 이미 잘 학습된 상태다.** 이 가중치는 같은 종류의 사진으로 충분히
학습된 것이라, 40장을 몇 번 더 본다고 나아지지 않는다. 이미 다 배운 학생에게
문제 몇 개 더 풀린 셈이다.

참고로 이 가중치의 실력은 이렇다. 그냥 확대하는 것보다 **0.8 dB** 좋다.

| | 시험 점수 |
|---|---|
| 그냥 확대 (bicubic) | 18.31 dB |
| 이 모델 | **19.10 dB** |

숫자가 안 오른다고 실패한 게 아니라, **올릴 여지가 없는 지점에서 시작**했기 때문이다.

## 5. 진짜 사진에 써보기

이제 인천 위성사진에 적용한다. 2001×2001 픽셀이 6003×6003으로 커진다.

**여기서는 방금 3회 돌린 결과를 쓰지 않는다.** 미리 제대로 학습해둔 가중치
(`edsr_ikonosfull_x3_latest.pt`)를 쓴다. 몇 초 학습으로는 쓸 만한 그림이 안 나온다.

지금까지 본 연습·시험 사진은 정답을 줄여 만든 것이었지만, **이 인천 사진은 진짜
Sentinel-2가 촬영한 것**이다. 실제로 쓰이는 상황이 이쪽이다. 대신 정답이 없어서
점수는 못 내고 눈으로 확인한다.

사진이 커서 한 번에 넣으면 메모리가 터진다. 그래서 작은 조각으로 잘라 처리하고
다시 이어붙인다. 조각 경계에 자국이 남지 않도록 살짝 겹쳐서 자른다.

결과는 위치 정보가 그대로 붙은 GeoTIFF라, QGIS 같은 지도 프로그램에서 원본 위에
바로 겹쳐볼 수 있다.

> `--chop` 옵션은 쓰지 마세요. 원본 코드에 버그가 있어서 켜면 에러가 납니다.

In [ ]:
OUTDIR = '/content/results'

# 방금 3회 돌린 결과가 아니라, 제대로 학습해둔 가중치를 쓴다.
# 몇 분짜리 학습으로는 아래 품질이 안 나온다.
weight = f'{MODEL}/checkpoints/edsr_ikonosfull_x3_latest.pt'

p = subprocess.run(['python', 'infer.py', '--weight', weight,
                    '--input', f'{DATA}/test', '--output', OUTDIR],
                   cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(p.stdout[-1500:])

In [ ]:
# 좌표가 보존됐는지 확인 — 실습에서 가장 자주 틀리는 부분이다
src_tif = glob.glob(f'{DATA}/test/*.tif')[0]
out_tif = glob.glob(f'{OUTDIR}/*_SRx3.tif')[0]

with rasterio.open(src_tif) as a, rasterio.open(out_tif) as b:
    print(f'입력  {a.width}x{a.height}  {a.res[0]:.4f} m  {a.crs}')
    print(f'출력  {b.width}x{b.height}  {b.res[0]:.4f} m  {b.crs}')
    same = np.allclose(np.array(a.bounds), np.array(b.bounds), atol=1e-6)
    print(f'\n지리 범위 일치: {same}')
    assert same, '좌표가 어긋났습니다'

## 6. 그냥 확대한 것과 비교하기

AI를 쓴 게 의미가 있었을까? 가장 단순한 확대 방법(bicubic, 주변 픽셀 평균내서 늘리기)과
비교해본다.

정답 사진이 없어서 점수는 못 낸다. 대신 두 가지로 판단한다.

- **선명도 수치(`lap_std`)** — 픽셀 사이 변화가 얼마나 급한지. 흐리면 낮고 또렷하면 높다.
- **눈으로 확인** — 건물 경계나 도로가 실제로 또렷해졌는지.

In [ ]:
import cv2

with rasterio.open(src_tif) as s:
    lr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
with rasterio.open(out_tif) as s:
    sr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
bic = cv2.resize(lr, (lr.shape[1] * 3, lr.shape[0] * 3), interpolation=cv2.INTER_CUBIC)

def lap_std(img):
    return float(cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'Bicubic x3   lap_std {lap_std(bic):6.2f}   평균밝기 {bic.reshape(-1,3).mean(0).round(1)}')
print(f'EDSR    x3   lap_std {lap_std(sr):6.2f}   평균밝기 {sr.reshape(-1,3).mean(0).round(1)}')
print('\n※ lap_std는 같은 격자끼리만 비교할 것. 10m 원본 LR의 값은 픽셀 계단 때문에 크게 나온다.')

In [ ]:
# 텍스처가 많은 구역을 골라 확대 비교
gray = cv2.cvtColor(sr, cv2.COLOR_RGB2GRAY)
S = 300
best, bs = (0, 0), -1
for y in range(0, sr.shape[0] - S, S):
    for x in range(0, sr.shape[1] - S, S):
        v = gray[y:y+S, x:x+S].std()
        if v > bs: best, bs = (y, x), v
y, x = best

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, img, t in zip(ax, [cv2.resize(lr[y//3:y//3+S//3, x//3:x//3+S//3], (S, S),
                                     interpolation=cv2.INTER_NEAREST), bic[y:y+S, x:x+S], sr[y:y+S, x:x+S]],
                     ['input LR (10 m, nearest)', 'Bicubic x3', 'EDSR x3']):
    a.imshow(img); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 7. 결과 저장하기

Colab이 꺼지면 파일이 사라진다. 남기고 싶은 건 구글 드라이브로 옮긴다.

In [ ]:
if SAVE_TO_DRIVE:
    dst = '/content/drive/MyDrive/sr_colab/results'
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{OUTDIR}/*'):
        shutil.copy(f, dst)
    print('저장:', os.listdir(dst))

---

## 자주 걸리는 함정

**1. `EPOCHS=10`을 넣으면 9번만 돈다.**
원본 코드가 그렇게 만들어져 있다. 10번 돌리고 싶으면 11을 넣자. `EPOCHS=1`은 아예
아무것도 배우지 않는다.

**2. `--chop` 옵션을 켜면 에러가 난다.**
원본 코드의 버그다. 큰 사진은 이 노트북처럼 조각내서 처리하면 된다.

**3. 자료를 바꿔 넣었는데 예전 것이 계속 나온다.**
`dataset/bin/` 폴더를 지우세요. 속도를 위해 사진을 미리 변환해 저장해두는데, 파일 이름이
같으면 예전 것을 그대로 쓴다.

**4. 학습이 이상하게 느리다.**
구글 드라이브에서 직접 학습시키면 안 된다. 드라이브는 인터넷 너머에 있어서 작은 파일을
많이 읽으면 아주 느려진다. 이 노트북처럼 Colab 안으로 복사한 뒤 학습해야 한다.

**5. 연결이 끊겨서 처음부터 다시 해야 한다.**
학습 폴더를 드라이브에 연결해뒀다면, `RESET`을 `'0'`으로 바꾸고 다시 실행하면 끊긴
지점부터 이어서 한다.

**6. 실전시험 점수가 낮은데 실패한 건가요?**
아니다. 실전시험은 진짜 위성이 찍은 다른 사진이라 원래 점수가 낮게 나온다.
중요한 건 절대값이 아니라 **그냥 확대한 것보다 나은가**이다.